# NESA — Fast Medical Knowledge Base Indexing

**Scope:** embeddings + ChromaDB dense index + BM25 sparse index + metadata.

This notebook does **not** implement hybrid fusion, reranking, or LLM generation.

### Local-first design
The default local model is `BAAI/bge-small-en-v1.5` rather than `bge-large-en-v1.5` to make CPU execution much faster. CUDA is used automatically when available. You can switch to BGE-base/large or OpenAI later.

Dense search handles paraphrased questions; BM25 protects exact medical terminology, drug names, dosages, units, acronyms, and guideline names.

## 1. Setup & Configuration

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Literal
import os, json, time, hashlib, pickle, logging, re

PROJECT_ROOT = Path(".").resolve()
CHUNKS_PATH = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\chunking\nesa_chunks.jsonl"
)

INDICES_DIR = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\INDICES"
)

VECTOR_DB_DIR = Path(
    r"D:\new\OneDrive\Desktop\cleaned_Data\VECTOR_DB"
)
BM25_INDEX_PATH = INDICES_DIR / "bm25_index.pkl"
EMBEDDING_CACHE_PATH = INDICES_DIR / "embedding_cache.npz"
EMBEDDING_CACHE_META_PATH = INDICES_DIR / "embedding_cache_meta.json"
INDICES_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class NesaConfig:
    embedding_backend: Literal["sentence_transformers", "openai"] = "sentence_transformers"
    st_model_name: str = "BAAI/bge-small-en-v1.5"
    openai_model: str = "text-embedding-3-small"
    openai_dimensions: int | None = None
    device: str | None = None
    local_batch_size: int = 128
    openai_batch_size: int = 64
    max_retries: int = 5
    initial_backoff_seconds: float = 1.0
    backoff_multiplier: float = 2.0
    collection_name: str = "nesa_medical_kb"
    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    text_field_for_embedding: str = "text"
    text_field_for_bm25: str = "text"
    use_embedding_cache: bool = True

CONFIG = NesaConfig()
print(CONFIG)
print("Chunks:", CHUNKS_PATH)
print("Indices:", INDICES_DIR)
print("Chroma:", VECTOR_DB_DIR)
print("BM25:", BM25_INDEX_PATH)

NesaConfig(embedding_backend='sentence_transformers', st_model_name='BAAI/bge-small-en-v1.5', openai_model='text-embedding-3-small', openai_dimensions=None, device=None, local_batch_size=128, openai_batch_size=64, max_retries=5, initial_backoff_seconds=1.0, backoff_multiplier=2.0, collection_name='nesa_medical_kb', bm25_k1=1.5, bm25_b=0.75, text_field_for_embedding='text', text_field_for_bm25='text', use_embedding_cache=True)
Chunks: D:\new\OneDrive\Desktop\cleaned_Data\chunking\nesa_chunks.jsonl
Indices: D:\new\OneDrive\Desktop\cleaned_Data\INDICES
Chroma: D:\new\OneDrive\Desktop\cleaned_Data\VECTOR_DB
BM25: D:\new\OneDrive\Desktop\cleaned_Data\INDICES\bm25_index.pkl


## 2. Imports & Client Initialization

In [2]:
import numpy as np
from tqdm.auto import tqdm
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("nesa_indexing")

openai_client = None
st_model = None
local_device = None

if CONFIG.embedding_backend == "sentence_transformers":
    from sentence_transformers import SentenceTransformer
    try:
        import torch
        local_device = CONFIG.device or ("cuda" if torch.cuda.is_available() else "cpu")
    except Exception:
        local_device = CONFIG.device or "cpu"
    print(f"Loading {CONFIG.st_model_name} on {local_device} ...")
    st_model = SentenceTransformer(CONFIG.st_model_name, device=local_device)
elif CONFIG.embedding_backend == "openai":
    from openai import OpenAI
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        raise EnvironmentError("OPENAI_API_KEY is missing.")
    openai_client = OpenAI(api_key=key)
else:
    raise ValueError(CONFIG.embedding_backend)

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR),
    settings=Settings(anonymized_telemetry=False),
)
print("Initialization complete.")

c:\Users\E_Magic\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BAAI/bge-small-en-v1.5 on cpu ...


2026-08-18 16:12:39,226 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-18 16:12:39,269 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-08-18 16:12:39,481 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-18 16:12:39,483 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-18 16:12:39,585 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-18 16:12:39,587 | INFO | Loading SentenceTransformer model from BAAI/b

Initialization complete.


## 3. Data Ingestion & Validation

In [3]:
def load_chunks(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(path)
    records=[]; bad=0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip(): continue
            try:
                x=json.loads(line)
                if isinstance(x, dict): records.append(x)
                else: bad += 1
            except json.JSONDecodeError:
                bad += 1
    if not records: raise ValueError("No valid chunks found")
    if bad: print(f"Warning: skipped {bad} malformed records")
    return records

def validate_chunks(chunks):
    seen=set(); dup=0; toks=[]; types={}
    for c in chunks:
        for k in ("chunk_id","document_id","text"):
            if not c.get(k): raise ValueError(f"Missing {k}: {c.get('chunk_id','<unknown>')}")
        cid=str(c["chunk_id"])
        if cid in seen: dup += 1
        seen.add(cid)
        if isinstance(c.get("token_count"),(int,float)): toks.append(int(c["token_count"]))
        dt=str(c.get("document_type") or "unknown"); types[dt]=types.get(dt,0)+1
    return {"total":len(chunks),"unique":len(seen),"duplicates":dup,
            "token_min":min(toks) if toks else None,"token_max":max(toks) if toks else None,
            "token_mean":float(np.mean(toks)) if toks else None,"types":types}

chunks=load_chunks(CHUNKS_PATH)
stats=validate_chunks(chunks)
print(json.dumps(stats, indent=2))

{
  "total": 13107,
  "unique": 13107,
  "duplicates": 0,
  "token_min": 35,
  "token_max": 20632,
  "token_mean": 260.45815213244833,
  "types": {
    "book": 13107
  }
}


## 4. Fast Embedding Engine

Uses the enriched `text` field. Embeddings are cached so a repeated run does not recompute an unchanged corpus/model.

In [4]:
def fingerprint(chunks, field, model_name):
    h=hashlib.sha256(model_name.encode())
    for c in chunks:
        h.update(str(c.get("chunk_id","")).encode()); h.update(b"\0")
        h.update(str(c.get(field) or c.get("raw_text") or "").encode()); h.update(b"\n")
    return h.hexdigest()

def load_cache(fp, n):
    if not CONFIG.use_embedding_cache or not EMBEDDING_CACHE_PATH.exists() or not EMBEDDING_CACHE_META_PATH.exists(): return None
    try:
        meta=json.loads(EMBEDDING_CACHE_META_PATH.read_text(encoding="utf-8"))
        if meta.get("fingerprint") != fp or meta.get("count") != n: return None
        x=np.load(EMBEDDING_CACHE_PATH)["embeddings"]
        if len(x)!=n: return None
        print("Using cached embeddings:", x.shape)
        return x.astype(np.float32)
    except Exception as e:
        print("Cache ignored:", e); return None

def save_cache(x, fp, model_name):
    np.savez_compressed(EMBEDDING_CACHE_PATH, embeddings=x)
    EMBEDDING_CACHE_META_PATH.write_text(json.dumps({"fingerprint":fp,"count":len(x),"model":model_name,"dimension":x.shape[1]},indent=2),encoding="utf-8")

def embed_openai(texts):
    out=[]
    for start in tqdm(range(0,len(texts),CONFIG.openai_batch_size),desc="OpenAI batches"):
        batch=texts[start:start+CONFIG.openai_batch_size]; delay=CONFIG.initial_backoff_seconds
        kwargs={"model":CONFIG.openai_model,"input":batch}
        if CONFIG.openai_dimensions: kwargs["dimensions"]=CONFIG.openai_dimensions
        for attempt in range(CONFIG.max_retries):
            try:
                r=openai_client.embeddings.create(**kwargs)
                out.extend(x.embedding for x in sorted(r.data,key=lambda z:z.index)); break
            except Exception:
                if attempt == CONFIG.max_retries-1: raise
                time.sleep(delay); delay*=CONFIG.backoff_multiplier
    return np.asarray(out,dtype=np.float32)

def generate_embeddings(chunks):
    field=CONFIG.text_field_for_embedding
    texts=[str(c.get(field) or c.get("raw_text") or "") for c in chunks]
    model_name=CONFIG.st_model_name if CONFIG.embedding_backend=="sentence_transformers" else CONFIG.openai_model
    fp=fingerprint(chunks,field,model_name)
    cached=load_cache(fp,len(chunks))
    if cached is not None: return cached
    t=time.time()
    if CONFIG.embedding_backend=="sentence_transformers":
        x=st_model.encode(texts,batch_size=CONFIG.local_batch_size,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
        x=np.asarray(x,dtype=np.float32)
    else:
        x=embed_openai(texts)
    if len(x)!=len(chunks): raise RuntimeError(f"Embedding count mismatch: {len(x)} vs {len(chunks)}")
    save_cache(x,fp,model_name)
    print(f"Generated {len(x):,} embeddings in {(time.time()-t)/60:.2f} min; dim={x.shape[1]}")
    return x

embedding_start=time.time()
embeddings=generate_embeddings(chunks)
embedding_elapsed=time.time()-embedding_start
embedding_dim=int(embeddings.shape[1])

Batches: 100%|██████████| 103/103 [36:38<00:00, 21.34s/it]


Generated 13,107 embeddings in 36.66 min; dim=384


## 5. ChromaDB Dense Vector Index

In [5]:
def chroma_metadata(c):
    def s(k,d=""): return str(c.get(k)) if c.get(k) is not None else d
    def i(k,d=-1): return int(c[k]) if isinstance(c.get(k),(int,float)) else d
    return {
        "chunk_id":s("chunk_id"),"document_id":s("document_id"),"source_file":s("source_file"),
        "document_type":s("document_type","unknown"),"title":s("title"),"page_start":i("page_start"),"page_end":i("page_end"),
        "chapter":s("chapter"),"section":s("section","unknown"),"content_type":s("content_type","text"),
        "token_count":i("token_count"),"raw_text":s("raw_text"),"text":s("text"),
        "source_block_indices_json":json.dumps(c.get("source_block_indices",[]),ensure_ascii=False),
        "cleaning_flags_json":json.dumps(c.get("cleaning_flags",[]),ensure_ascii=False),
    }

try: chroma_client.delete_collection(CONFIG.collection_name)
except Exception: pass
collection=chroma_client.get_or_create_collection(CONFIG.collection_name,metadata={"hnsw:space":"cosine"})

for start in tqdm(range(0,len(chunks),256),desc="Chroma upsert batches"):
    batch=chunks[start:start+256]; vec=embeddings[start:start+256]
    collection.upsert(
        ids=[str(c["chunk_id"]) for c in batch],
        embeddings=vec.tolist(),
        documents=[str(c.get("text") or c.get("raw_text") or "") for c in batch],
        metadatas=[chroma_metadata(c) for c in batch],
    )

assert collection.count()==len(chunks)
print("Chroma records:", collection.count())

Chroma upsert batches: 100%|██████████| 52/52 [00:33<00:00,  1.54it/s]

Chroma records: 13107


## 6. BM25 Indexing & Persistence

In [6]:
try:
    from nltk.stem import PorterStemmer
    stemmer=PorterStemmer()
except Exception:
    stemmer=None

STOPWORDS={"a","an","the","and","or","but","is","are","was","were","be","been","being","in","on","at","to","for","with","as","by","this","that","these","those","it","its","from","which","who","whom"}
TOKEN_PATTERN=re.compile(r"[a-z0-9]+(?:[.\-/][a-z0-9]+)*",re.I)

def tokenize_medical(text):
    result=[]
    for tok in TOKEN_PATTERN.findall((text or "").lower()):
        numeric=bool(re.search(r"\d",tok))
        if not numeric and tok in STOPWORDS: continue
        if stemmer and not numeric: tok=stemmer.stem(tok)
        result.append(tok)
    return result

corpus=[str(c.get(CONFIG.text_field_for_bm25) or c.get("raw_text") or "") for c in chunks]
tokenized=[tokenize_medical(x) for x in tqdm(corpus,desc="BM25 tokenization")]
bm25_index=BM25Okapi(tokenized,k1=CONFIG.bm25_k1,b=CONFIG.bm25_b)
bm25_doc_id_mapping=[str(c["chunk_id"]) for c in chunks]
chunk_lookup={str(c["chunk_id"]):c for c in chunks}

with BM25_INDEX_PATH.open("wb") as f:
    pickle.dump({"bm25_index":bm25_index,"doc_id_mapping":bm25_doc_id_mapping,"chunk_lookup":chunk_lookup},f,pickle.HIGHEST_PROTOCOL)

with BM25_INDEX_PATH.open("rb") as f: check=pickle.load(f)
assert len(check["doc_id_mapping"])==len(chunks)
print("BM25 saved:", BM25_INDEX_PATH)

BM25 tokenization: 100%|██████████| 13107/13107 [00:05<00:00, 2600.30it/s]


BM25 saved: D:\new\OneDrive\Desktop\cleaned_Data\INDICES\bm25_index.pkl


## 7. Retrieval Verification & Metadata Filtering

In [7]:
def query_embedding(query):
    if CONFIG.embedding_backend=="sentence_transformers":
        return np.asarray(st_model.encode([query],normalize_embeddings=True,convert_to_numpy=True)[0],dtype=np.float32)
    kwargs={"model":CONFIG.openai_model,"input":[query]}
    if CONFIG.openai_dimensions: kwargs["dimensions"]=CONFIG.openai_dimensions
    return np.asarray(openai_client.embeddings.create(**kwargs).data[0].embedding,dtype=np.float32)

def dense_search(query,top_k=5,where=None):
    kwargs={"query_embeddings":[query_embedding(query).tolist()],"n_results":top_k,"include":["metadatas","documents","distances"]}
    if where: kwargs["where"]=where
    r=collection.query(**kwargs); out=[]
    for cid,d,m,doc in zip(r["ids"][0],r["distances"][0],r["metadatas"][0],r["documents"][0]):
        out.append({"chunk_id":cid,"score":round(1-float(d),4),"document_type":m.get("document_type"),"section":m.get("section"),"page_start":m.get("page_start"),"preview":(doc or "")[:220]})
    return out

def bm25_search(query,top_k=5):
    scores=bm25_index.get_scores(tokenize_medical(query)); idx=np.argsort(scores)[::-1][:top_k]; out=[]
    for i in idx:
        c=chunk_lookup[bm25_doc_id_mapping[int(i)]]
        out.append({"chunk_id":bm25_doc_id_mapping[int(i)],"score":round(float(scores[i]),4),"document_type":c.get("document_type"),"section":c.get("section"),"page_start":c.get("page_start"),"preview":str(c.get("raw_text") or c.get("text") or "")[:220]})
    return out

def show(title,results):
    print("\n"+"="*90+"\n"+title+"\n"+"="*90)
    for n,r in enumerate(results,1): print(f"{n}. {r['score']} | {r['chunk_id']} | {r['document_type']} | p.{r['page_start']}\n   {r['preview']}...")

q1="What are the warning signs of postpartum depression?"
show("Dense — Postpartum",dense_search(q1)); show("BM25 — Postpartum",bm25_search(q1))

q2="WHO Labour Care Guide guidelines for stage 1 labor monitoring."
show("Dense — Research",dense_search(q2)); show("BM25 — Research",bm25_search(q2))

types=sorted({str(c.get("document_type") or "unknown") for c in chunks})
print("\nAvailable document types:",types)
if "patient_education" in types:
    show("Filtered Dense — patient_education",dense_search(q1,where={"document_type":"patient_education"}))
else:
    print("No patient_education chunks found; filtered test skipped.")

Batches: 100%|██████████| 1/1 [00:00<00:00, 20.43it/s]



Dense — Postpartum
1. 0.8387 | c14a4045c893_chunk_0070 | book | p.40
   [Document Type: book] [Chapter: AT THE TIME OF THE EVENT] [Page: 40]

. sadness, tearfulness, irritability and anxiety), insomnia and decreased concentration.

The symptoms of postpartum blues develop within two to three...
2. 0.8387 | 6efc90007443_chunk_0070 | book | p.40
   [Document Type: book] [Chapter: AT THE TIME OF THE EVENT] [Page: 40]

. sadness, tearfulness, irritability and anxiety), insomnia and decreased concentration.

The symptoms of postpartum blues develop within two to three...
3. 0.8387 | c14a4045c893_chunk_0071 | book | p.42
   [Document Type: book] [Chapter: POSTPARTUM DEPRESSION] [Page: 42]

Postpartum depression affects up to 34% of women. It typically occurs in the early postpartum weeks or months and may persist for a year or more.

Depres...
4. 0.8387 | 6efc90007443_chunk_0071 | book | p.42
   [Document Type: book] [Chapter: POSTPARTUM DEPRESSION] [Page: 42]

Postpartum depression affects

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.00it/s]


Dense — Research
1. 0.8472 | 13ce3bfd1981_chunk_0007 | book | p.9
   [Document Type: book] [Section: Introduction] [Page: 9]

WHO recommendations on intrapartum care specify evidence-based practices that should be implemented throughout labour and the immediate postnatal periods, and disc...
2. 0.8304 | ce2e100ba161_chunk_0028 | book | p.11
   [Document Type: book] [Chapter: ISBN 978-92-4-155021-5] [Section: WHO] [Page: 11]

Recommendations The GDG agreed that, to achieve a positive The WHO technical consultations led to 56 childbirth experience for women and ...
3. 0.8304 | e6675c575145_chunk_0028 | book | p.11
   [Document Type: book] [Chapter: ISBN 978-92-4-155021-5] [Section: WHO] [Page: 11]

Recommendations The GDG agreed that, to achieve a positive The WHO technical consultations led to 56 childbirth experience for women and ...
4. 0.8304 | 046522535872_chunk_0028 | book | p.11
   [Document Type: book] [Chapter: ISBN 978-92-4-155021-5] [Section: WHO] [Page: 11]

Recommendations 

## 8. Summary & Readiness

In [8]:
total=time.time() if False else embedding_elapsed
summary={
    "chunks":len(chunks),
    "embedding_backend":CONFIG.embedding_backend,
    "embedding_model":CONFIG.st_model_name if CONFIG.embedding_backend=="sentence_transformers" else CONFIG.openai_model,
    "device":local_device,
    "embedding_dimension":embedding_dim,
    "chroma_records":collection.count(),
    "bm25_records":len(bm25_doc_id_mapping),
    "chroma_path":str(VECTOR_DB_DIR),
    "bm25_path":str(BM25_INDEX_PATH),
    "embedding_cache":str(EMBEDDING_CACHE_PATH),
    "hybrid_retrieval_ready":collection.count()==len(chunks)==len(bm25_doc_id_mapping),
    "reranking_implemented":False,
    "generation_implemented":False,
}
print(json.dumps(summary,indent=2))
print("\nREADY for downstream Hybrid Retrieval/Reranking." if summary["hybrid_retrieval_ready"] else "\nIndex consistency check FAILED.")

{
  "chunks": 13107,
  "embedding_backend": "sentence_transformers",
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "device": "cpu",
  "embedding_dimension": 384,
  "chroma_records": 13107,
  "bm25_records": 13107,
  "chroma_path": "D:\\new\\OneDrive\\Desktop\\cleaned_Data\\VECTOR_DB",
  "bm25_path": "D:\\new\\OneDrive\\Desktop\\cleaned_Data\\INDICES\\bm25_index.pkl",
  "embedding_cache": "D:\\new\\OneDrive\\Desktop\\cleaned_Data\\INDICES\\embedding_cache.npz",
  "hybrid_retrieval_ready": true,
  "reranking_implemented": false,
  "generation_implemented": false
}

READY for downstream Hybrid Retrieval/Reranking.
